<a href="https://colab.research.google.com/github/rudalshan0412-code/Intent_Classifier-RAG_Chatbot/blob/main/05)_%EC%A7%88%EB%AC%B8%EC%97%90_%EB%8C%80%ED%95%9C_%EB%8B%B5%EB%B3%80_%EC%83%9D%EC%84%B1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 구글 드라이브 연결
from google.colab import drive

drive.mount("/content/drive")

%cd /content/drive/MyDrive/rag_intent_chatbot

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/rag_intent_chatbot


In [ ]:
# chatbot.py

%%writefile src/chatbot.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Any

from src.rag.retriever import Retriever
from src.rag.vector_store import SearchResult


@dataclass
class ChatbotResult:
    """사용자 입력 한 건의 처리 결과를 저장한다."""

    user_input: str
    predicted_intent: str
    confidence: float
    is_fallback: bool
    requires_rag: bool
    response: str
    search_results: list[SearchResult]


class Chatbot:
    """Intent 예측 결과에 따라 일반 응답과 RAG 검색을 분기한다."""

    def __init__(
        self,
        intent_predictor: Any, # clas IntentPredicotr 아니면 FakeIntentPredictor 가 들어온다
        retriever: Retriever,
        confidence_threshold: float = 0.60,
        retrieval_top_k: int = 3,
        preview_length: int = 200,
    ) -> None:
        if not 0.0 <= confidence_threshold <= 1.0:
            raise ValueError(
                "confidence_threshold는 0 이상 1 이하여야 합니다."
            )

        if retrieval_top_k < 1:
            raise ValueError(
                "retrieval_top_k는 1 이상이어야 합니다."
            )

        if preview_length < 1:
            raise ValueError(
                "preview_length는 1 이상이어야 합니다."
            )

        self.intent_predictor = intent_predictor
        self.retriever = retriever
        self.confidence_threshold = confidence_threshold
        self.retrieval_top_k = retrieval_top_k
        self.preview_length = preview_length

    def process_message(
        self,
        user_input: str,
    ) -> ChatbotResult:
        """사용자 문장을 분류하고 알맞은 처리 경로로 전달한다."""

        if not isinstance(user_input, str):
            raise TypeError("user_input은 문자열이어야 합니다.")

        user_input = user_input.strip()

        if not user_input:
            return ChatbotResult(
                user_input=user_input,
                predicted_intent="fallback",
                confidence=0.0,
                is_fallback=True,
                requires_rag=False,
                response="질문을 입력해주세요.",
                search_results=[],
            )

        prediction = self.intent_predictor.predict(
            text=user_input,
            threshold=self.confidence_threshold,
            top_k=3,
        )

        predicted_intent = prediction["intent"]
        confidence = float(prediction["confidence"])
        is_fallback = predicted_intent == "fallback"
        requires_rag = bool(prediction["requires_rag"])

        if is_fallback:
            return ChatbotResult(
                user_input=user_input,
                predicted_intent=predicted_intent,
                confidence=confidence,
                is_fallback=True,
                requires_rag=False,
                response=prediction["response"],
                search_results=[],
            )

        if not requires_rag:
            return ChatbotResult(
                user_input=user_input,
                predicted_intent=predicted_intent,
                confidence=confidence,
                is_fallback=False,
                requires_rag=False,
                response=prediction["response"],
                search_results=[],
            )

        search_results = self.retriever.retrieve(
            query=user_input,
            top_k=self.retrieval_top_k,
        )

        response = self._format_search_results(search_results)

        return ChatbotResult(
            user_input=user_input,
            predicted_intent=predicted_intent,
            confidence=confidence,
            is_fallback=False,
            requires_rag=True,
            response=response,
            search_results=search_results,
        )

    def _format_search_results(
        self,
        search_results: list[SearchResult],
    ) -> str:
        """검색 결과를 읽기 쉬운 문자열로 변환한다."""

        if not search_results:
            return "관련 문서 내용을 찾지 못했습니다."

        lines = ["문서에서 다음과 같은 관련 내용을 찾았습니다."]

        for result in search_results:
            preview = result.chunk.text[:self.preview_length]

            if len(result.chunk.text) > self.preview_length:
                preview += "..."

            lines.extend(
                [
                    "",
                    f"[검색 결과 {result.rank}]",
                    f"유사도: {result.score:.4f}",
                    f"출처: {result.chunk.source}",
                    f"Chunk ID: {result.chunk.chunk_id}",
                    f"내용: {preview}",
                ]
            )

        return "\n".join(lines)

Overwriting src/chatbot.py


In [ ]:
# 파일 확인

!sed -n '1,260p' src/chatbot.py

from __future__ import annotations

from dataclasses import dataclass
from typing import Any

from src.rag.retriever import Retriever
from src.rag.vector_store import SearchResult


@dataclass
class ChatbotResult:
    """사용자 입력 한 건의 처리 결과를 저장한다."""

    user_input: str
    predicted_intent: str
    confidence: float
    is_fallback: bool
    requires_rag: bool
    response: str
    search_results: list[SearchResult]


class Chatbot:
    """Intent 예측 결과에 따라 일반 응답과 RAG 검색을 분기한다."""

    def __init__(
        self,
        intent_predictor: Any,
        retriever: Retriever,
        confidence_threshold: float = 0.60,
        retrieval_top_k: int = 3,
        preview_length: int = 200,
    ) -> None:
        if not 0.0 <= confidence_threshold <= 1.0:
            raise ValueError(
                "confidence_threshold는 0 이상 1 이하여야 합니다."
            )

        if retrieval_top_k < 1:
            raise ValueError(
                "retrieval_top_k는 1 이상이어야 합니다."
            )

        if prev

In [ ]:
# 가짜 개별 테스트(Predictor 기반)

# 가짜 Predictor, Retriever 생성

from src.chatbot import Chatbot
from src.rag.chunker import Chunk
from src.rag.vector_store import SearchResult

class FakeIntentPredictor:
    def predict(
        self,
        text: str,
        threshold: float = 0.60,
        top_k: int = 3,
    ) -> dict:
        fake_predictions = {
            "안녕하세요": {
                "intent": "greeting",
                "confidence": 0.95,
                "response": "안녕하세요. 무엇을 도와드릴까요?",
            },
            "고마워요": {
                "intent": "thanks",
                "confidence": 0.92,
                "response": "도움이 되었다니 다행입니다.",
            },
            "임베딩은 무엇인가요?": {
                "intent": "document_query",
                "confidence": 0.90,
                "response": "문서를 검색하겠습니다.",
            },
            "알 수 없는 문장": {
                "intent": "greeting",
                "confidence": 0.30,
                "response": "인사 응답",
            },
        }

        result = fake_predictions[text]

        if result["confidence"] < threshold:
            predicted_intent = "fallback"
            response = (
                "질문의 의도를 확실하게 판단하지 못했습니다. "
                "조금 더 구체적으로 질문해주세요."
            )
        else:
            predicted_intent = result["intent"]
            response = result["response"]

        return {
            "text": text,
            "intent": predicted_intent,
            "confidence": result["confidence"],
            "response": response,
            "top_predictions": [],
            "requires_rag": predicted_intent == "document_query",
        }


class FakeRetriever:
    def __init__(self) -> None:
        self.call_count = 0

    def retrieve(
        self,
        query: str,
        top_k: int = 3,
    ) -> list[SearchResult]:
        self.call_count += 1

        fake_chunk = Chunk(
            text=(
                "임베딩은 텍스트와 같은 데이터를 "
                "숫자 벡터 형태로 표현하는 방법입니다."
            ),
            chunk_id="sample_chunk_0000",
            metadata={"source": "sample.txt"},
            source="sample.txt",
            start_index=0,
            end_index=41,
        )

        return [
            SearchResult(
                chunk=fake_chunk,
                score=0.91,
                rank=1,
            )
        ]

In [ ]:
# 가짜 Chatbot 생성

fake_predictor = FakeIntentPredictor()
fake_retriever = FakeRetriever()

fake_chatbot = Chatbot(
    intent_predictor=fake_predictor,
    retriever=fake_retriever,
    confidence_threshold=0.60,
    retrieval_top_k=3,
)

In [ ]:
# 각 분기 테스트

test_messages = [
    "안녕하세요",
    "고마워요",
    "임베딩은 무엇인가요?",
    "알 수 없는 문장",
]

for message in test_messages:
    result = fake_chatbot.process_message(message)

    print("=" * 60)
    print(f"사용자 입력: {result.user_input}")
    print(f"predicted_intent: {result.predicted_intent}")
    print(f"confidence: {result.confidence:.4f}")
    print(f"fallback 여부: {result.is_fallback}")
    print(f"requires_rag: {result.requires_rag}")
    print(f"검색 결과 개수: {len(result.search_results)}")
    print(f"응답:\n{result.response}")

사용자 입력: 안녕하세요
predicted_intent: greeting
confidence: 0.9500
fallback 여부: False
requires_rag: False
검색 결과 개수: 0
응답:
안녕하세요. 무엇을 도와드릴까요?
사용자 입력: 고마워요
predicted_intent: thanks
confidence: 0.9200
fallback 여부: False
requires_rag: False
검색 결과 개수: 0
응답:
도움이 되었다니 다행입니다.
사용자 입력: 임베딩은 무엇인가요?
predicted_intent: document_query
confidence: 0.9000
fallback 여부: False
requires_rag: True
검색 결과 개수: 1
응답:
문서에서 다음과 같은 관련 내용을 찾았습니다.

[검색 결과 1]
유사도: 0.9100
출처: sample.txt
Chunk ID: sample_chunk_0000
내용: 임베딩은 텍스트와 같은 데이터를 숫자 벡터 형태로 표현하는 방법입니다.
사용자 입력: 알 수 없는 문장
predicted_intent: fallback
confidence: 0.3000
fallback 여부: True
requires_rag: False
검색 결과 개수: 0
응답:
질문의 의도를 확실하게 판단하지 못했습니다. 조금 더 구체적으로 질문해주세요.


In [ ]:
# 호출 횟수 확인

print("Retriever 호출 횟수:", fake_retriever.call_count)

# 정상적이라면 호출 횟수는 1번이여야한다(document_query는 한 문장이기에 Retrier 또한 1번만 실행)

Retriever 호출 횟수: 1


In [ ]:
# 자동 검증

greeting_result = fake_chatbot.process_message("안녕하세요")
thanks_result = fake_chatbot.process_message("고마워요")
document_result = fake_chatbot.process_message("임베딩은 무엇인가요?")
fallback_result = fake_chatbot.process_message("알 수 없는 문장")

assert greeting_result.predicted_intent == "greeting"
assert greeting_result.requires_rag is False
assert len(greeting_result.search_results) == 0

assert thanks_result.predicted_intent == "thanks"
assert thanks_result.requires_rag is False
assert len(thanks_result.search_results) == 0

assert document_result.predicted_intent == "document_query"
assert document_result.requires_rag is True
assert len(document_result.search_results) == 1

assert fallback_result.predicted_intent == "fallback"
assert fallback_result.is_fallback is True
assert fallback_result.requires_rag is False
assert len(fallback_result.search_results) == 0

print("가짜 객체 기반 라우팅 테스트 통과")

# 이미 4번 테스트를 한 상태임으로 call_count는 누적된다.
# 만약 호출 횟수를 정확히 1로 검증하고 싶은 경우 가짜 객체를 다시 생성해야한다.

가짜 객체 기반 라우팅 테스트 통과


In [ ]:
# 실제 Intent Predictor 초기화

from src.intent.predict import IntentPredictor

intent_predictor = IntentPredictor(
    model_path="models/intent_classifier.pt",
)

# 정상적으로 로딩됐는지 확인
test_prediction = intent_predictor.predict(
    text="안녕하세요",
    threshold=0.60,
    top_k=3,
)

test_prediction

'''예상 구조

{
    "text": "안녕하세요",
    "intent": "greeting",
    "confidence": ...,
    "response": "...",
    "top_predictions": [...],
    "requires_rag": False
}
'''

'예상 구조\n\n{\n    "text": "안녕하세요",\n    "intent": "greeting",\n    "confidence": ...,\n    "response": "...",\n    "top_predictions": [...],\n    "requires_rag": False\n}\n'

In [ ]:
# 실제 RAG Retriever 초기화

from src.rag.document_loader import Document, load_documents
from src.rag.text_preprocessor import preprocess_text
from src.rag.chunker import chunk_documents
from src.rag.embedder import TextEmbedder
from src.rag.vector_store import VectorStore
from src.rag.retriever import Retriever

# 문서 로딩
documents = load_documents(
    directory_path="data/documents",
)

print("문서 개수:", len(documents))

# 문서 전처리

preprocessed_documents = []

for document in documents:
    preprocessed_document = Document(
        text=preprocess_text(document.text),
        metadata=document.metadata,
    )

    preprocessed_documents.append(preprocessed_document)

print("전처리된 문서 개수:", len(preprocessed_documents))

문서 개수: 1
전처리된 문서 개수: 1


In [ ]:
# Chunk 생성

chunks = chunk_documents(
    documents=preprocessed_documents,
    chunk_size=500,
    chunk_overlap=100,
)

print("생성된 Chunk 개수:", len(chunks))

# Chunk 하나 확인

print("Chunk ID:", chunks[0].chunk_id)
print("출처:", chunks[0].source)
print("내용:")
print(chunks[0].text[:300])

생성된 Chunk 개수: 9
Chunk ID: sample_chunk_0000
출처: sample.txt
내용:
RAG의 정의

RAG는 Retrieval-Augmented Generation의 약자로, 검색 증강 생성이라고 부른다. 일반적인 생성형 언어 모델은 학습 과정에서 익힌 지식과 현재 입력된 문맥을 바탕으로 답변을 만든다. 반면 RAG는 사용자의 질문과 관련된 외부 문서를 먼저 검색하고, 검색된 내용을 언어 모델의 입력 문맥에 함께 제공한 뒤 답변을 생성한다. 따라서 모델이 모든 정보를 내부 파라미터에 기억하고 있지 않더라도 프로젝트 문서, 수업 자료, 매뉴얼, 보고서와 같은 별도의 지식 저장소를 활용할 수 있다. 이 프로젝트에서는 


In [ ]:
# 임베딩 모델 생성

embedder = TextEmbedder()

print("임베딩 모델:", embedder.model_name)
print("실행 장치:", embedder.device)
print("임베딩 차원:", embedder.embedding_dimension)

# Chunk 임베딩

chunk_texts = [
    chunk.text
    for chunk in chunks
]

chunk_embeddings = embedder.encode_texts(
    texts=chunk_texts,
)

print("임베딩 배열 크기:", chunk_embeddings.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/content/drive/MyDrive/rag_intent_chatbot/src/rag/embedder.py:29: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dimension = self.model.get_sentence_embedding_dimension() # 출력하는 벡터(임베딩)의 차원 수(크기)를 반환


임베딩 모델: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
실행 장치: cuda:0
임베딩 차원: 384
임베딩 배열 크기: (9, 384)


In [ ]:
# VectorStore 생성

vector_store = VectorStore()

vector_store.add(
    chunks=chunks,
    embeddings=chunk_embeddings,
)

print("VectorStore Chunk 개수:", len(vector_store.chunks))
print("VectorStore 임베딩 크기:", vector_store.embeddings.shape)

VectorStore Chunk 개수: 9
VectorStore 임베딩 크기: (9, 384)


In [ ]:
# Retriever 생성

retriever = Retriever(
    embedder=embedder,
    vector_store=vector_store,
)

# Retriever 확인

retrieval_results = retriever.retrieve(
    query="임베딩은 무엇인가요?",
    top_k=3,
)

for result in retrieval_results:
    print("=" * 50)
    print("rank:", result.rank)
    print("score:", round(result.score, 4))
    print("chunk_id:", result.chunk.chunk_id)
    print("source:", result.chunk.source)
    print("text:", result.chunk.text[:200])

rank: 1
score: 0.3504
chunk_id: sample_chunk_0008
source: sample.txt
text: 프롬프트에 넣어 근거 중심의 응답을 만들 수 있다. 현재 단계에서는 이 전체 구조 중 가장 앞부분인 문서 로딩, 공백 및 줄바꿈 전처리, 문단과 문장을 고려한 청킹을 구현한다. 이후에는 임베딩 생성, 벡터 저장소, Retriever, Intent Classifier와 RAG 라우팅, 최종 챗봇 인터페이스 순서로 확장할 예정이다.
rank: 2
score: 0.3425
chunk_id: sample_chunk_0003
source: sample.txt
text: 너무 긴 문단이나 문장은 최종적으로 글자 수 기준으로 나눌 수 있으며, 인접 Chunk 사이에 일부 내용을 겹치게 두면 경계 부근의 문맥이 사라지는 문제를 완화할 수 있다.

임베딩의 의미

컴퓨터는 문장의 의미를 사람처럼 직접 이해하지 못하므로, 텍스트를 수치 벡터로 변환하는 과정이 필요하다. 임베딩은 단어, 문장, 문서 조각의 의미적 특징을 여러 차원의
rank: 3
score: 0.3012
chunk_id: sample_chunk_0006
source: sample.txt
text: Intent Classifier는 사용자의 문장을 greeting, goodbye, thanks, help, bot_info, document_query 중 하나로 분류한다. 최고 예측 확률이 임계값보다 낮으면 fallback으로 처리하며, document_query로 예측된 경우에만 requires_rag 값을 참으로 설정한다. 이 구조를 사용하면 단순 


In [ ]:
# 전체 통합 테스트

from src.chatbot import Chatbot

chatbot = Chatbot(
    intent_predictor=intent_predictor,
    retriever=retriever,
    confidence_threshold=0.60,
    retrieval_top_k=3,
    preview_length=200,
)

# 통합 테스트 문장

integration_test_messages = [
    "안녕하세요",
    "고마워요",
    "너는 누구야?",
    "사용 방법을 알려줘",
    "임베딩은 무엇인가요?",
    "문서를 왜 Chunk로 나누나요?",
    "파란 생각이 조용하게 숫자를 걸어간다", # 의미가 모호한 문장
]

# 전체 결과 출력

for message in integration_test_messages:
    result = chatbot.process_message(message)

    print("\n" + "=" * 70)
    print(f"사용자 입력: {result.user_input}")
    print(f"predicted_intent: {result.predicted_intent}")
    print(f"confidence: {result.confidence:.4f}")
    print(f"fallback 여부: {result.is_fallback}")
    print(f"requires_rag: {result.requires_rag}")
    print(f"검색 결과 개수: {len(result.search_results)}")
    print(f"응답:\n{result.response}")

    if result.requires_rag:
        print("\n[검색 결과 상세]")

        for search_result in result.search_results:
            print("-" * 50)
            print(f"rank: {search_result.rank}")
            print(f"score: {search_result.score:.4f}")
            print(f"chunk_id: {search_result.chunk.chunk_id}")
            print(f"source: {search_result.chunk.source}")
            print(
                "Chunk 내용:",
                search_result.chunk.text[:200],
            )


사용자 입력: 안녕하세요
predicted_intent: greeting
confidence: 0.9540
fallback 여부: False
requires_rag: False
검색 결과 개수: 0
응답:
반갑습니다. 궁금한 내용을 입력해주세요.

사용자 입력: 고마워요
predicted_intent: fallback
confidence: 0.3180
fallback 여부: True
requires_rag: False
검색 결과 개수: 0
응답:
질문의 의도를 확실하게 판단하지 못했습니다. 조금 더 구체적으로 질문해주세요.

사용자 입력: 너는 누구야?
predicted_intent: bot_info
confidence: 0.9903
fallback 여부: False
requires_rag: False
검색 결과 개수: 0
응답:
현재 프로젝트에서는 PyTorch 기반 인텐트 분류와 문서 검색 기능을 구현하고 있습니다.

사용자 입력: 사용 방법을 알려줘
predicted_intent: help
confidence: 0.9453
fallback 여부: False
requires_rag: False
검색 결과 개수: 0
응답:
일반적인 대화를 하거나 저장된 문서에 관해 질문할 수 있습니다.

사용자 입력: 임베딩은 무엇인가요?
predicted_intent: fallback
confidence: 0.3180
fallback 여부: True
requires_rag: False
검색 결과 개수: 0
응답:
질문의 의도를 확실하게 판단하지 못했습니다. 조금 더 구체적으로 질문해주세요.

사용자 입력: 문서를 왜 Chunk로 나누나요?
predicted_intent: document_query
confidence: 0.7030
fallback 여부: False
requires_rag: True
검색 결과 개수: 3
응답:
문서에서 다음과 같은 관련 내용을 찾았습니다.

[검색 결과 1]
유사도: 0.5267
출처: sample.txt
Chunk ID: sample_c